In [5]:
from typing import Optional
import xarray as xr
import pandas as pd
import numpy as np
import cftime
import plotly.express as px
from scipy.stats import skew, kurtosis

In [6]:
DATA_PATH = r"C:\Users\jsieg\Documents\4dvd-clone\backend\dataset\GPCP V2.3 Precipitation\GPCP V2.3 Precipitation_Single Level\Monthly Error Estimate (Surface)\Monthly Error Estimate (Surface).nc"

In [15]:
from typing import Optional
import xarray as xr
import pandas as pd
import numpy as np
import cftime
import plotly.express as px
from scipy.stats import skew, kurtosis
DATA_PATH = r"C:\Users\jsieg\Documents\4dvd-clone\backend\dataset\GPCP V2.3 Precipitation\GPCP V2.3 Precipitation_Single Level\Monthly Error Estimate (Surface)\Monthly Error Estimate (Surface).nc"
# --- Imports ---
from typing import Optional
import xarray as xr
import pandas as pd
import numpy as np
import cftime
import plotly.express as px

# --- Config ---


# ---------------- Utils ----------------
def iso_times_from_coord(time_coord) -> list[str]:
    """Convert xarray time coordinates to ISO strings."""
    vals = time_coord.values
    out = []
    vals_list = [vals] if vals.shape == () else list(vals)
    for t in vals_list:
        if isinstance(t, (np.datetime64, pd.Timestamp)):
            out.append(pd.to_datetime(t).strftime("%Y-%m-%d"))
        elif hasattr(t, 'year') and hasattr(t, 'month') and hasattr(t, 'day'):
            out.append(f"{t.year:04d}-{t.month:02d}-{t.day:02d}")
        else:
            out.append(str(t))
    return out


def guess_cmap_name(varname: str, units: str) -> str:
    """Suggest a colormap based on variable name and units."""
    name = varname.lower()
    u = (units or "").lower()
    if any(k in name for k in ["anom", "anomaly", "difference"]):
        return "coolwarm"
    if any(k in u for k in ["k", "°c", "degc", "kelvin", "celsius", "temperature"]):
        return "turbo"
    if any(k in name for k in ["precip", "rain", "snow", "snod", "pr"]):
        return "viridis"
    if any(k in name for k in ["wind", "speed"]):
        return "plasma"
    return "viridis"


def choose_best_variable(ds: xr.Dataset, fallback: str = "precip") -> str:
    """Pick the most appropriate variable in the dataset."""
    variables = list(ds.data_vars)

    # Remove "bounds"-style variables
    filtered = [v for v in variables if "bnds" not in v.lower()]

    if not filtered:
        return variables[0]  # if nothing left, fallback to whatever exists

    # 1. Explicit fallback
    if fallback in filtered:
        return fallback

    # 2. Prioritize by content (similar to guess_cmap_name)
    for v in filtered:
        name = v.lower()
        units = (ds[v].attrs.get("units") or "").lower()

        if any(k in name for k in ["anom", "anomaly", "difference"]):
            return v
        if any(k in units for k in ["k", "°c", "degc", "kelvin", "celsius", "temperature"]):
            return v
        if any(k in name for k in ["precip", "rain", "snow", "snod", "pr"]):
            return v
        if any(k in name for k in ["wind", "speed"]):
            return v

    # 3. Otherwise just take the first filtered variable
    return filtered[0]


def open_dataset_flexible(path: str, variable_of_interest: Optional[str] = None):
    """Open NetCDF file, pick a variable, and detect multi-level structure."""
    ds = xr.open_dataset(path, decode_times=True, use_cftime=True)

    # Fix time parsing
    if "time" in ds.coords:
        time_vals = ds["time"].values
        if isinstance(time_vals[0], (cftime.DatetimeNoLeap, cftime.DatetimeGregorian)):
            pass
        elif pd.api.types.is_numeric_dtype(time_vals):
            units = ds["time"].attrs.get("units", "days since 1800-01-01")
            ref_date = units.split("since")[-1].strip()
            ds["time"] = pd.to_datetime(ref_date) + pd.to_timedelta(time_vals, unit="D")

    # Pick variable
    if variable_of_interest and variable_of_interest in ds.data_vars:
        chosen_var = variable_of_interest
    else:
        chosen_var = choose_best_variable(ds)

    da = ds[chosen_var]

    # ✅ Detect multi-level
    multilevel = False
    levels = None
    if "level" in da.dims:
        multilevel = True
        levels = da["level"].values
    elif "plev" in da.dims:
        multilevel = True
        levels = da["plev"].values

    return ds, chosen_var, multilevel, levels


def is_multilevel(da: xr.DataArray) -> bool:
    """Detect if a DataArray has vertical levels (pressure or height)."""
    return "level" in da.dims or "plev" in da.dims

def select_time_safe(da, date_str: str):
    """Select a time slice safely for both cftime and pandas time coords."""
    try:
        target = pd.to_datetime(date_str)
        time_vals = da["time"].values
        if isinstance(time_vals[0], cftime.DatetimeGregorian):
            # Convert to cftime
            target = cftime.DatetimeGregorian(target.year, target.month, target.day)
        return da.sel(time=target, method="nearest")
    except Exception as e:
        print("⚠️ Fallback to first timestep due to:", e)
        return da.isel(time=0)
    
def compute_point_statistics(
    da: xr.DataArray,
    lat: float,
    lon: float,
    level: Optional[float] = None,
    date: Optional[str] = None
) -> pd.DataFrame:
    """
    Compute descriptive statistics for a single point (lat, lon, level).
    If date is provided, compute stats for that timestep only.
    Otherwise, compute stats across the whole time series.
    """

    # Select nearest grid point
    point = da.sel(lat=lat, lon=lon, method="nearest")

    # Handle multilevel datasets
    if is_multilevel(da):
        if level is None:
            level = float(point["level"].values[0])  # default = first available
        point = point.sel(level=level, method="nearest")

    # Handle time selection
    if date:
        point = select_time_safe(point, date)

    arr = point.values.flatten().astype(float)
    arr = arr[np.isfinite(arr)]  # drop NaNs

    if len(arr) == 0:
        return pd.DataFrame([{}], index=["Empty"])

    stats = {
        "Min": np.min(arr),
        "25%": np.percentile(arr, 25),
        "50%": np.median(arr),
        "Mean": np.mean(arr),
        "75%": np.percentile(arr, 75),
        "Max": np.max(arr),
        "Std": np.std(arr, ddof=0),
        "Var": np.var(arr, ddof=0),
        "Skewness": skew(arr, bias=False),
        "Kurtosis": kurtosis(arr, bias=False),
    }

    return pd.DataFrame([stats], index=[f"({lat:.2f}, {lon:.2f})"])
    
def compute_point_stats(da: xr.DataArray, lat: float, lon: float, level: Optional[float] = None) -> pd.DataFrame:
    # Select nearest grid point
    point = da.sel(lat=lat, lon=lon, method="nearest")

    # If multi-level, pick level
    if is_multilevel(da):
        if level is None:
            level = float(point["level"].values[0])  # default to first available
        point = point.sel(level=level, method="nearest")

    # Extract time values
    times = iso_times_from_coord(point["time"])
    vals = point.values.astype(float)

    # Drop NaNs
    mask = np.isfinite(vals)
    times = np.array(times)[mask]
    vals = vals[mask]

    # Compute stats
    mean_val = np.mean(vals)
    std_val = np.std(vals, ddof=0)

    df = pd.DataFrame({
        "Time": times,
        "Value": vals,
    })
    df["Mean"] = mean_val
    df["Std"] = std_val

    return df.set_index("Time")

import plotly.express as px

def plot_point_timeseries(da: xr.DataArray, lat: float, lon: float, level: Optional[float] = None):
    # Select nearest grid point
    point = da.sel(lat=lat, lon=lon, method="nearest")

    # Handle multi-level datasets
    if is_multilevel(da):
        if level is None:
            level = float(point["level"].values[0])  # default = first available
        point = point.sel(level=level, method="nearest")

    times = iso_times_from_coord(point["time"])
    vals = point.values.astype(float)

    # Drop NaNs
    mask = np.isfinite(vals)
    times = np.array(times)[mask]
    vals = vals[mask]

    fig = px.line(
        x=times, y=vals,
        title=f"Time Series at ({lat}N, {lon}E)" + (f", level={level}" if is_multilevel(da) else ""),
        labels={"x": "Time", "y": f"{point.name} ({point.attrs.get('units','')})"}
    )
    return fig


def plot_point_histogram(da: xr.DataArray, lat: float, lon: float, level: Optional[float] = None, bins: int = 30):
    # Select nearest grid point
    point = da.sel(lat=lat, lon=lon, method="nearest")

    # Handle multi-level datasets
    if is_multilevel(da):
        if level is None:
            level = float(point["level"].values[0])
        point = point.sel(level=level, method="nearest")

    vals = point.values.astype(float)
    vals = vals[np.isfinite(vals)]  # drop NaNs

    fig = px.histogram(
        x=vals, nbins=bins,
        title=f"Histogram at ({lat}N, {lon}E)" + (f", level={level}" if is_multilevel(da) else ""),
        labels={"x": f"{point.name} ({point.attrs.get('units','')})", "y": "Frequency"},
    )
    return fig

def plot_point_histogram_month(
    da: xr.DataArray,
    lat: float,
    lon: float,
    year: int,
    month: int,
    level: Optional[float] = None,
    bins: int = 30
):
    # Select nearest grid point
    point = da.sel(lat=lat, lon=lon, method="nearest")

    # Handle multi-level
    if is_multilevel(da):
        if level is None:
            level = float(point["level"].values[0])
        point = point.sel(level=level, method="nearest")

    # Slice by month
    start = pd.Timestamp(year=year, month=month, day=1)
    end = start + pd.offsets.MonthEnd(1)
    point = point.sel(time=slice(start, end))

    # Flatten values
    vals = point.values.astype(float)
    vals = vals[np.isfinite(vals)]

    fig = px.histogram(
        x=vals, nbins=bins,
        title=f"Histogram at ({lat}N, {lon}E) for {year}-{month:02d}" + (f", level={level}" if is_multilevel(da) else ""),
        labels={"x": f"{point.name} ({point.attrs.get('units','')})", "y": "Frequency"},
    )
    return fig

def plot_point_month_histogram_across_years(
    da: xr.DataArray,
    lat: float,
    lon: float,
    month: int,
    level: Optional[float] = None,
    bins: int = 20
):
    """
    Histogram of a chosen month's values (e.g. all Januaries across years)
    at a specific (lat, lon) grid point.
    """

    # Select nearest grid point
    point = da.sel(lat=lat, lon=lon, method="nearest")

    # Handle multi-level datasets
    if is_multilevel(da):
        if level is None:
            level = float(point["level"].values[0])
        point = point.sel(level=level, method="nearest")

    # Convert time coordinate to pandas datetime for easy filtering
    times = pd.to_datetime(iso_times_from_coord(point["time"]))
    vals = point.values.astype(float)

    # Drop NaNs
    mask = np.isfinite(vals)
    times = times[mask]
    vals = vals[mask]

    # Filter values for the chosen month across all years
    month_mask = times.month == month
    month_vals = vals[month_mask]

    if len(month_vals) == 0:
        raise ValueError(f"No data available for month={month}")

    # Build title
    month_name = pd.to_datetime(f"2000-{month:02d}-01").strftime("%B")
    dataset_name = da.name or "Variable"
    title = (
        f"{month_name} Histogram of {dataset_name} at "
        f"{'Single Level' if not is_multilevel(da) else f'Level {level}'} "
        f"(Lat: {float(point['lat'].values):.2f}° N, Lon: {float(point['lon'].values):.2f}° E)"
    )

    # Plot histogram
    fig = px.histogram(
        x=month_vals, nbins=bins,
        title=title,
        labels={"x": f"{dataset_name} ({da.attrs.get('units','')})", "y": "Frequency"},
    )
    return fig

def compute_monthly_mean_std(
    da: xr.DataArray,
    lat: float,
    lon: float,
    year: int,
    level: Optional[float] = None
) -> pd.DataFrame:
    """
    Compute mean and standard deviation for each month of a given year
    at a specific grid point (lat, lon, level).
    Works for single-level and multi-level datasets.
    """

    # Select nearest grid point
    point = da.sel(lat=lat, lon=lon, method="nearest")

    # Handle multi-level datasets
    if is_multilevel(da):
        if level is None:
            level = float(point["level"].values[0])  # default = first available
        point = point.sel(level=level, method="nearest")

    # Convert time to pandas for easy filtering
    times = pd.to_datetime(iso_times_from_coord(point["time"]))
    vals = point.values.astype(float)

    # Drop NaNs
    mask = np.isfinite(vals)
    times = times[mask]
    vals = vals[mask]

    # Create DataFrame of time/value
    df = pd.DataFrame({"Time": times, "Value": vals})
    df["Year"] = df["Time"].dt.year
    df["Month"] = df["Time"].dt.month

    # Filter for selected year
    df_year = df[df["Year"] == year]
    if df_year.empty:
        raise ValueError(f"No data available for year {year}")

    # Compute mean + std for each month
    stats = df_year.groupby("Month")["Value"].agg(
        Mean="mean", Std="std"
    ).reset_index()

    # Add month name for readability
    stats["Month Name"] = stats["Month"].apply(lambda m: pd.to_datetime(f"2000-{m:02d}-01").strftime("%B"))

    return stats.set_index("Month Name")[["Mean", "Std"]]

import pandas as pd
import numpy as np

def compute_monthly_mean_yearly_std(
    da, lat: float, lon: float, year: int, level: float = None
) -> pd.DataFrame:
    """
    Compute monthly mean for a specific year,
    but standard deviation across all years (population std, ddof=0).
    """

    # Select nearest grid point
    point = da.sel(lat=lat, lon=lon, method="nearest")

    # Handle multi-level datasets
    if "level" in point.dims or "plev" in point.dims:
        if level is None:
            level = float(point["level"].values[0])
        point = point.sel(level=level, method="nearest")

    # Convert time safely
    times = pd.to_datetime(iso_times_from_coord(point["time"]))
    vals = point.values.astype(float)

    # Drop NaNs
    mask = np.isfinite(vals)
    times = times[mask]
    vals = vals[mask]

    # Build dataframe
    df = pd.DataFrame({"Time": times, "Value": vals})
    df["Year"] = df["Time"].dt.year
    df["Month"] = df["Time"].dt.month

    # --- Mean for the chosen year ---
    df_year = df[df["Year"] == year]
    means = df_year.groupby("Month")["Value"].mean()

    # --- Std across *all years* ---
    stds = df.groupby("Month")["Value"].agg(lambda x: np.std(x, ddof=0))

    # Combine
    stats = pd.DataFrame({"Mean": means, "Std": stds})
    stats = stats.reset_index()

    # Add month names
    stats["Month Name"] = stats["Month"].apply(
        lambda m: pd.to_datetime(f"2000-{m:02d}-01").strftime("%B")
    )

    return stats.set_index("Month Name")[["Mean", "Std"]]

def compute_seasonal_timeseries(
    da, lat: float, lon: float, month: int, start_year: int, end_year: int, level: float = None
) -> pd.DataFrame:
    """
    Extract seasonal time series: values for a specific month across multiple years
    at a given (lat, lon, level).
    Example: January values from 1980–2020.
    """

    # Select nearest grid point
    point = da.sel(lat=lat, lon=lon, method="nearest")

    # Handle multi-level datasets
    if "level" in point.dims or "plev" in point.dims:
        if level is None:
            level = float(point["level"].values[0])
        point = point.sel(level=level, method="nearest")

    # Convert time safely
    times = pd.to_datetime(iso_times_from_coord(point["time"]))
    vals = point.values.astype(float)

    # Drop NaNs
    mask = np.isfinite(vals)
    times = times[mask]
    vals = vals[mask]

    # Build dataframe
    df = pd.DataFrame({"Time": times, "Value": vals})
    df["Year"] = df["Time"].dt.year
    df["Month"] = df["Time"].dt.month

    # Filter by selected month and year range
    df_season = df[(df["Month"] == month) & (df["Year"].between(start_year, end_year))]

    if df_season.empty:
        raise ValueError(f"No data available for month={month} between {start_year}–{end_year}")

    # Keep only year + value
    seasonal_series = df_season[["Year", "Value"]].set_index("Year")

    return seasonal_series

import plotly.express as px

def plot_seasonal_timeseries(
    da, lat: float, lon: float,
    month: int, start_year: int, end_year: int,
    level: float = None
):
    """
    Plot a seasonal time series (e.g., all Januaries across years) at (lat, lon, level).
    Uses Plotly Express for interactive visualization.
    """

    # Select nearest grid point
    point = da.sel(lat=lat, lon=lon, method="nearest")

    # Handle multi-level datasets
    if "level" in point.dims or "plev" in point.dims:
        if level is None:
            level = float(point["level"].values[0])  # default first level
        point = point.sel(level=level, method="nearest")

    # Convert times safely
    times = pd.to_datetime(iso_times_from_coord(point["time"]))
    vals = point.values.astype(float)

    # Drop NaNs
    mask = np.isfinite(vals)
    times = times[mask]
    vals = vals[mask]

    # Build dataframe
    df = pd.DataFrame({"Time": times, "Value": vals})
    df["Year"] = df["Time"].dt.year
    df["Month"] = df["Time"].dt.month

    # Filter to chosen month + year range
    df_season = df[(df["Month"] == month) & (df["Year"].between(start_year, end_year))]

    if df_season.empty:
        raise ValueError(f"No data for month={month} between {start_year}–{end_year}")

    # Plot
    month_name = pd.to_datetime(f"2000-{month:02d}-01").strftime("%B")
    title = f"{month_name} Seasonal Time Series at ({lat}N, {lon}E)"
    if "level" in point.dims or "plev" in point.dims:
        title += f" (Level={level})"

    fig = px.line(
        df_season,
        x="Year", y="Value",
        markers=True,
        title=title,
        labels={"Value": f"{da.name} ({da.attrs.get('units', '')})"}
    )

    return fig

import plotly.express as px

def plot_monthly_climatology(
    da, lat: float, lon: float, level: float = None
):
    """
    Plot monthly climatology (mean), historical high, and historical low
    at a given (lat, lon, level).
    """

    # Select nearest grid point
    point = da.sel(lat=lat, lon=lon, method="nearest")

    # Handle multi-level datasets
    if "level" in point.dims or "plev" in point.dims:
        if level is None:
            level = float(point["level"].values[0])
        point = point.sel(level=level, method="nearest")

    # Convert time safely
    times = pd.to_datetime(iso_times_from_coord(point["time"]))
    vals = point.values.astype(float)

    # Drop NaNs
    mask = np.isfinite(vals)
    times = times[mask]
    vals = vals[mask]

    # Build DataFrame
    df = pd.DataFrame({"Time": times, "Value": vals})
    df["Month"] = df["Time"].dt.month

    # Compute climatology stats
    clim = df.groupby("Month")["Value"].agg(
        Climatology="mean",
        HistoricalHigh="max",
        HistoricalLow="min"
    ).reset_index()

    # Add month names
    clim["Month Name"] = clim["Month"].apply(
        lambda m: pd.to_datetime(f"2000-{m:02d}-01").strftime("%B")
    )

    # Build dynamic title
    dataset_name = da.name or "Variable"
    dataset_units = da.attrs.get("units", "")
    title = (
        f"{dataset_name} | Monthly Climatology "
        f"(Lat: {lat:.2f}° N, Lon: {lon:.2f}° W"
        + (f", Level: {level}" if "level" in da.dims or "plev" in da.dims else "")
        + ")"
    )

    # --- Melt to long format for plotting ---
    plot_df = clim.melt(
        id_vars=["Month Name"],
        value_vars=["Climatology", "HistoricalHigh", "HistoricalLow"],
        var_name="LineType",
        value_name="Value"
    )

    # Plot
    fig = px.line(
        plot_df,
        x="Month Name", y="Value", color="LineType",
        markers=True,
        title=title,
        labels={"Value": f"{dataset_name} ({dataset_units})", "Month Name": "Month"}
    )

    return fig, clim


In [3]:
# Open dataset
ds, var, multilevel, levels = open_dataset_flexible(DATA_PATH)
da = ds[var]

lat, lon = 28.75, -113.75  # pick a location
level = float(levels[0]) if multilevel else None

# Plot histogram for ALL Januaries across dataset (1979–2023)
fig = plot_point_month_histogram_across_years(da, lat=lat, lon=lon, month=1, level=level, bins=15)
fig.show()


C:\Users\jsieg\AppData\Local\Temp\ipykernel_42640\1187124222.py:85: DeprecationWarning: Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)

  ds = xr.open_dataset(path, decode_times=True, use_cftime=True)


In [4]:
ds, var, multilevel, levels = open_dataset_flexible(DATA_PATH)
da = ds[var]

lat, lon = 28.75, -113.75
level = float(levels[0]) if multilevel else None

stats_df = compute_point_statistics(da, lat, lon, level=level)
print(stats_df)

                       Min       25%       50%      Mean       75%       Max  \
(28.75, -113.75)  0.091944  0.127566  0.148645  0.163933  0.184172  0.455506   

                       Std       Var  Skewness  Kurtosis  
(28.75, -113.75)  0.055663  0.003098  2.013706  5.823263  


C:\Users\jsieg\AppData\Local\Temp\ipykernel_42640\1187124222.py:85: DeprecationWarning:

Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)




In [5]:
ds, var, multilevel, levels = open_dataset_flexible(DATA_PATH)
da = ds[var]

lat, lon = 28.75, -113.75
level = float(levels[0]) if multilevel else None

# Compute stats for year 1980
stats_1980 = compute_monthly_mean_yearly_std(da, lat=28.75, lon=-113.75, year=1980, level=level)
print(stats_1980)



                Mean       Std
Month Name                    
January     0.267103  0.072313
February    0.240444  0.051341
March       0.313878  0.060796
April       0.260470  0.069675
May         0.157193  0.033329
June        0.172859  0.029889
July        0.179631  0.023405
August      0.190071  0.031564
September   0.214721  0.055696
October     0.210585  0.068408
November    0.371179  0.052349
December    0.301242  0.058719


C:\Users\jsieg\AppData\Local\Temp\ipykernel_42640\1187124222.py:85: DeprecationWarning:

Usage of 'use_cftime' as a kwarg is deprecated. Please pass a 'CFDatetimeCoder' instance initialized with 'use_cftime' to the 'decode_times' kwarg instead.
Example usage:
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(decode_times=time_coder)




In [9]:
fig = plot_seasonal_timeseries(
    da, lat=30, lon=-115,
    month=1,  # January
    start_year=1980, end_year=2014,
    level=level  # only needed if multi-level dataset
)
fig.show()


In [17]:
fig, clim_stats = plot_monthly_climatology(
    da, lat=28.75, lon=-111.25, level=level
)
fig.show()
